# House Price Prediction - Complete Upgrade

## Features:
- Data Cleaning & Missing Value Handling
- Advanced Feature Engineering
- Multiple ML Models (Ridge, Lasso, Random Forest, Gradient Boosting, XGBoost)
- Cross-Validation & Model Comparison
- Feature Importance Analysis
- Hyperparameter Tuning
- Interactive Dashboard with Streamlit

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('All libraries loaded successfully!')

## 2. Load Data

In [ ]:
df = pd.read_csv('housing.csv')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

## 3. Data Cleaning

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')
print(f'Rows before cleaning: {len(df)}')

df['total_bedrooms'].fillna(df['total_bedrooms'].median(), inplace=True)

print(f'Rows after cleaning: {len(df)}')
print('Missing values after cleaning:')
print(df.isnull().sum())

In [ ]:
df.info()
print('\n')
df.describe().round(2)

## 4. Data Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

axes[0,0].hist(df['median_house_value'], bins=50, edgecolor='black', color='steelblue')
axes[0,0].set_title('House Value Distribution', fontsize=14)
axes[0,0].set_xlabel('Price ($)')

axes[0,1].scatter(df['median_income'], df['median_house_value'], alpha=0.2, color='coral')
axes[0,1].set_title('Income vs House Value', fontsize=14)
axes[0,1].set_xlabel('Median Income')
axes[0,1].set_ylabel('House Value')

axes[0,2].scatter(df['total_rooms'], df['median_house_value'], alpha=0.2, color='seagreen')
axes[0,2].set_title('Rooms vs House Value', fontsize=14)
axes[0,2].set_xlabel('Total Rooms')

df['ocean_proximity'].value_counts().plot(kind='bar', ax=axes[1,0], color='mediumpurple')
axes[1,0].set_title('Ocean Proximity Distribution', fontsize=14)
axes[1,0].tick_params(axis='x', rotation=45)

axes[1,1].hist(df['housing_median_age'], bins=30, edgecolor='black', color='goldenrod')
axes[1,1].set_title('Housing Age Distribution', fontsize=14)
axes[1,1].set_xlabel('Age')

axes[1,2].scatter(df['population'], df['median_house_value'], alpha=0.2, color='teal')
axes[1,2].set_title('Population vs House Value', fontsize=14)
axes[1,2].set_xlabel('Population')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

numeric_cols = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm', center=0, ax=axes[0], fmt='.2f')
axes[0].set_title('Correlation Heatmap', fontsize=14)

ocean_stats = df.groupby('ocean_proximity')['median_house_value'].agg(['mean', 'median']).sort_values('mean')
ocean_stats.plot(kind='bar', ax=axes[1], color=['#2196F3', '#FF9800'])
axes[1].set_title('Price by Ocean Proximity', fontsize=14)
axes[1].set_ylabel('Price ($)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df.boxplot(column='median_house_value', by='ocean_proximity', ax=axes[0])
axes[0].set_title('Price Distribution by Ocean Proximity', fontsize=14)
axes[0].set_xlabel('Ocean Proximity')
axes[0].set_ylabel('Price ($)')
plt.sca(axes[0])
plt.xticks(rotation=45)

df['income_bracket'] = pd.cut(df['median_income'], bins=[0, 2, 4, 6, 8, np.inf],
                               labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
income_stats = df.groupby('income_bracket', observed=True)['median_house_value'].mean()
income_stats.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Average Price by Income Bracket', fontsize=14)
axes[1].set_ylabel('Average Price ($)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
df['rooms_per_household'] = df['total_rooms'] / df['households']
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']
df['population_per_household'] = df['population'] / df['households']

print('New features created:')
print('- rooms_per_household')
print('- bedrooms_per_room')
print('- population_per_household')
print(f'\nDataset shape: {df.shape}')
df[['total_rooms', 'households', 'rooms_per_household', 'total_bedrooms', 
    'bedrooms_per_room', 'population', 'population_per_household']].describe().round(2)

In [ ]:
correlations = df.select_dtypes(include=[np.number]).corr()['median_house_value'].sort_values(ascending=False)

plt.figure(figsize=(10, 6))
correlations.drop('median_house_value').plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Feature Correlations with House Value', fontsize=14)
plt.xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()

print('\nCorrelation with median_house_value:')
print(correlations.round(4))

## 6. Prepare Data

In [ ]:
X = df.drop(['median_house_value', 'income_bracket'], axis=1)
y = df['median_house_value']

numeric_features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
                    'total_bedrooms', 'population', 'households', 'median_income',
                    'rooms_per_household', 'bedrooms_per_room', 'population_per_household']
categorical_features = ['ocean_proximity']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features)
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Test set: {X_test.shape[0]:,} samples')
print(f'Features: {X.shape[1]}')

## 7. Model Training & Comparison

In [ ]:
models = {
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=1.0, max_iter=10000),
    'Random Forest': RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=150, random_state=42, verbosity=0, n_jobs=-1)
}

results = {}
trained_pipelines = {}

for name, model in models.items():
    print(f'Training {name}...')
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2')
    
    results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
        'MAPE (%)': mape,
        'CV_R2_Mean': cv_scores.mean(),
        'CV_R2_Std': cv_scores.std()
    }
    trained_pipelines[name] = pipeline
    
    print(f'  R2: {r2:.4f} | RMSE: ${rmse:,.0f} | MAE: ${mae:,.0f} | MAPE: {mape:.1f}%')
    print(f'  CV R2: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
    print()

## 8. Model Comparison Visualization

In [ ]:
results_df = pd.DataFrame(results).T

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

results_df['R2'].plot(kind='bar', ax=axes[0,0], color='steelblue', edgecolor='black')
axes[0,0].set_title('R2 Score (Higher is Better)', fontsize=14)
axes[0,0].set_ylabel('R2 Score')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].set_ylim(0, 1)

results_df['RMSE'].plot(kind='bar', ax=axes[0,1], color='coral', edgecolor='black')
axes[0,1].set_title('RMSE (Lower is Better)', fontsize=14)
axes[0,1].set_ylabel('RMSE ($)')
axes[0,1].tick_params(axis='x', rotation=45)

results_df['MAE'].plot(kind='bar', ax=axes[1,0], color='seagreen', edgecolor='black')
axes[1,0].set_title('MAE (Lower is Better)', fontsize=14)
axes[1,0].set_ylabel('MAE ($)')
axes[1,0].tick_params(axis='x', rotation=45)

x_pos = range(len(results_df))
axes[1,1].bar(x_pos, results_df['CV_R2_Mean'], yerr=results_df['CV_R2_Std'], 
              color='mediumpurple', edgecolor='black', capsize=5)
axes[1,1].set_xticks(x_pos)
axes[1,1].set_xticklabels(results_df.index, rotation=45)
axes[1,1].set_title('Cross-Validation R2 (Mean +/- Std)', fontsize=14)
axes[1,1].set_ylabel('R2 Score')

plt.tight_layout()
plt.show()

In [ ]:
print('Model Comparison Results:')
print('='*80)
results_df.style.format({
    'RMSE': '${:,.0f}', 'MAE': '${:,.0f}', 
    'R2': '{:.4f}', 'MAPE (%)': '{:.1f}',
    'CV_R2_Mean': '{:.4f}', 'CV_R2_Std': '{:.4f}'
}).highlight_max(subset=['R2', 'CV_R2_Mean'], color='lightgreen').highlight_min(subset=['RMSE', 'MAE', 'MAPE (%)'], color='lightgreen')

## 9. Best Model Selection & Feature Importance

In [ ]:
best_model_name = results_df['R2'].idxmax()
best_r2 = results_df.loc[best_model_name, 'R2']
print(f'Best Model: {best_model_name}')
print(f'R2 Score: {best_r2:.4f}')
print(f'RMSE: ${results_df.loc[best_model_name, "RMSE"]:,.0f}')

best_pipeline = trained_pipelines[best_model_name]

ohe_features = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
all_features = list(numeric_features) + list(ohe_features)

if hasattr(best_pipeline.named_steps['regressor'], 'feature_importances_'):
    importances = best_pipeline.named_steps['regressor'].feature_importances_
    feat_imp = pd.Series(importances, index=all_features).sort_values(ascending=True)
    
    plt.figure(figsize=(10, 6))
    feat_imp.plot(kind='barh', color='steelblue', edgecolor='black')
    plt.title(f'Feature Importance - {best_model_name}', fontsize=14)
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
else:
    if hasattr(best_pipeline.named_steps['regressor'], 'coef_'):
        coef = best_pipeline.named_steps['regressor'].coef_
        feat_imp = pd.Series(np.abs(coef), index=all_features).sort_values(ascending=True)
        
        plt.figure(figsize=(10, 6))
        feat_imp.plot(kind='barh', color='steelblue', edgecolor='black')
        plt.title(f'Feature Coefficients (Absolute) - {best_model_name}', fontsize=14)
        plt.xlabel('|Coefficient|')
        plt.tight_layout()
        plt.show()

## 10. Actual vs Predicted Analysis

In [ ]:
y_pred = best_pipeline.predict(X_test)
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_test, y_pred, alpha=0.3, color='steelblue')
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title('Actual vs Predicted', fontsize=14)

axes[1].hist(residuals, bins=50, edgecolor='black', color='coral')
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual ($)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution', fontsize=14)

axes[2].scatter(y_pred, residuals, alpha=0.3, color='seagreen')
axes[2].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[2].set_xlabel('Predicted Price ($)')
axes[2].set_ylabel('Residual ($)')
axes[2].set_title('Residuals vs Predicted', fontsize=14)

plt.tight_layout()
plt.show()

print(f'Mean Residual: ${residuals.mean():,.0f}')
print(f'Std Residual: ${residuals.std():,.0f}')

## 11. Save Best Model

In [ ]:
joblib.dump(best_pipeline, 'best_house_price_model.pkl')

model_meta = {
    'features': list(X.columns),
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'results': results,
    'best_name': best_model_name,
    'metrics': results[best_model_name]
}
joblib.dump(model_meta, 'model_features.pkl')

print(f'Model saved: best_house_price_model.pkl')
print(f'Metadata saved: model_features.pkl')
print(f'Best model: {best_model_name} (R2: {best_r2:.4f})')